In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ['HF_HOME'] = '/home/jovyan/kuratov/data/.cache/huggingface'
os.chdir('/home/jovyan/kuratov/data/test_time_gd/')

import torch

In [1]:


from transformers import AutoModel

model = AutoModel.from_pretrained('gpt2')

print(model)

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)


In [2]:
import transformers
transformers.__version__

'4.51.3'

In [3]:
import torch
torch.__version__


'2.6.0+cu124'

In [4]:
# torch package path:
torch.__file__

'/home/jovyan/kuratov/envs/py311_pt2.6_cu12.4/lib/python3.11/site-packages/torch/__init__.py'

In [5]:
import vllm
vllm.__version__

INFO 05-08 01:12:29 [__init__.py:239] Automatically detected platform cuda.


'0.8.4'

In [5]:
from kv_dataset_utils import create_tokenizer
tokenizer = create_tokenizer()

In [ ]:
from transformers import AutoConfig
config = AutoConfig.from_pretrained('EleutherAI/pythia-160m')
config.num_hidden_layers = 4
config.num_attention_heads = 4
config.hidden_size = 128
config.vocab_size = 128
config.intermediate_size = config.hidden_size * 4
config.pad_token_id = tokenizer.convert_tokens_to_ids('[PAD]')
config.bos_token_id = tokenizer.convert_tokens_to_ids('[BOS]')
config.eos_token_id = tokenizer.convert_tokens_to_ids('[EOS]')
config

GPTNeoXConfig {
  "architectures": [
    "GPTNeoXForCausalLM"
  ],
  "attention_bias": true,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "classifier_dropout": 0.1,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout": 0.0,
  "hidden_size": 128,
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 2048,
  "model_type": "gpt_neox",
  "num_attention_heads": 4,
  "num_hidden_layers": 4,
  "pad_token_id": 0,
  "partial_rotary_factor": 0.25,
  "rope_scaling": null,
  "rope_theta": 10000,
  "rotary_emb_base": 10000,
  "rotary_pct": 0.25,
  "tie_word_embeddings": false,
  "torch_dtype": "float16",
  "transformers_version": "4.51.3",
  "use_cache": true,
  "use_parallel_residual": true,
  "vocab_size": 128
}

In [43]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_config(config)

In [56]:
model

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(128, 128)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-3): 4 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (query_key_value): Linear(in_features=128, out_features=384, bias=True)
          (dense): Linear(in_features=128, out_features=128, bias=True)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=128, out_features=512, bias=True)
          (dense_4h_to_h): Linear(in_features=512, out_features=128, bias=True)
          (act): GELUActivation()
        )
      )
    )
    (final_layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affi

In [51]:
input_ids = tokenizer('|ABCDEF|', return_tensors='pt')

In [57]:
out = model(**input_ids, labels=input_ids['input_ids'])

In [58]:
out.keys()

odict_keys(['loss', 'logits', 'past_key_values'])

In [59]:
out.logits.shape

torch.Size([1, 8, 128])

In [75]:
model = AutoModelForCausalLM.from_config(config)
model.to(torch.bfloat16)

out = model(**input_ids, labels=input_ids['input_ids'])
print(out.loss)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-04)
optimizer.zero_grad()
out.loss.backward()
optimizer.step()


with torch.no_grad():
    out = model(**input_ids, labels=input_ids['input_ids'])
    print(out.loss)

tensor(4.8883, grad_fn=<NllLossBackward0>)
tensor(4.3410)


In [79]:
model.config.model_type

'gpt_neox'

In [80]:
model

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(128, 128)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-3): 4 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (query_key_value): Linear(in_features=128, out_features=384, bias=True)
          (dense): Linear(in_features=128, out_features=128, bias=True)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=128, out_features=512, bias=True)
          (dense_4h_to_h): Linear(in_features=512, out_features=128, bias=True)
          (act): GELUActivation()
        )
      )
    )
    (final_layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affi

In [81]:
model.gpt_neox

GPTNeoXModel(
  (embed_in): Embedding(128, 128)
  (emb_dropout): Dropout(p=0.0, inplace=False)
  (layers): ModuleList(
    (0-3): 4 x GPTNeoXLayer(
      (input_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (post_attention_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (post_attention_dropout): Dropout(p=0.0, inplace=False)
      (post_mlp_dropout): Dropout(p=0.0, inplace=False)
      (attention): GPTNeoXAttention(
        (query_key_value): Linear(in_features=128, out_features=384, bias=True)
        (dense): Linear(in_features=128, out_features=128, bias=True)
      )
      (mlp): GPTNeoXMLP(
        (dense_h_to_4h): Linear(in_features=128, out_features=512, bias=True)
        (dense_4h_to_h): Linear(in_features=512, out_features=128, bias=True)
        (act): GELUActivation()
      )
    )
  )
  (final_layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (rotary_emb): GPTNeoXRotaryEmbedding()
)

tensor([[[-0.0170, -0.0432,  0.0082,  ..., -0.0093, -0.0177,  0.0028],
         [-0.0250,  0.0354,  0.0165,  ...,  0.0134,  0.0182, -0.0297],
         [ 0.0040,  0.0089,  0.0050,  ..., -0.0025, -0.0176, -0.0113],
         ...,
         [-0.0435,  0.0160, -0.0070,  ...,  0.0060,  0.0117, -0.0063],
         [ 0.0038, -0.0095, -0.0153,  ...,  0.0220, -0.0190,  0.0269],
         [-0.0170, -0.0432,  0.0082,  ..., -0.0093, -0.0177,  0.0028]]],
       dtype=torch.bfloat16, grad_fn=<EmbeddingBackward0>)